# COM Sweep Special Point Analysis

This notebook investigates the suspected special point near symmetric COM offset `-0.35`.

The analysis is split into two parts:

1. **Fine local continuation scan** over `[-0.36, -0.34]` to decide whether the eigenvalue change is smooth but steep, or truly discontinuous.
2. **Full physical free-run comparison** on both sides of the suspected jump to inspect impacts, releases, phase portraits, and geometry.

## 1. Imports and Setup

In [ ]:
import importlib
import sys
from pathlib import Path

sys.dont_write_bytecode = True

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "passive_brachiation").exists():
    PROJECT_ROOT = Path(r"c:\Users\82646\OneDrive\桌面\Chalmers work\p4\TRA455 Artistic Intelligence in Robotics\Project\2026-05-23-mini_project_passive_brachiation")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np

import passive_brachiation.simulation as simulation_module
import passive_brachiation.shooting as shooting_module
import passive_brachiation.com_continuation as com_continuation_module

# Keep this notebook robust when rerun in a live kernel after editing the package.
simulation_module = importlib.reload(simulation_module)
shooting_module = importlib.reload(shooting_module)
com_continuation_module = importlib.reload(com_continuation_module)

from passive_brachiation import (
    BrachiationParameters,
    BrachiationState,
    CollisionMode,
    Slope,
    SwitchDecision,
    TwoLinkBrachiationModel,
    continue_fixed_point_branch,
    evaluate_elbow_below_slope_section,
    ik_from_stride_distance,
    make_passive_brachiation_feasibility_check,
    make_passive_brachiation_stride_map,
    parameters_with_symmetric_com_offset,
    release_indices,
    samples_to_arrays,
    scan_stride_fixed_points,
    simulate,
)

plt.rcParams["figure.figsize"] = (8, 5)
print("Project root:", PROJECT_ROOT)

In [ ]:
params = BrachiationParameters.uniform_links(
    m1=1.041,
    m2=1.041,
    l1=0.314,
    l2=0.314,
    damping1=0.0,
    damping2=0.0,
    gravity=9.81,
)
model = TwoLinkBrachiationModel(params)
GAMMA = np.deg2rad(45.0)
slope = Slope(gamma=GAMMA)
INITIAL_SUPPORT = np.zeros(2, dtype=float)

# Reference values used only for display in this setup cell.
REFERENCE_BEST_COM_OFFSET = 1.0 / 3.0
REFERENCE_SPECIAL_OFFSETS = (-0.34, -0.358)

def switch_policy(t, state, support_point, impact_point, slope):
    return SwitchDecision.SWITCH

shoot_dt = 0.005
shoot_t_max = 8.0
initial_direction = -1.0
impact_direction = 1.0

def print_com_offset(label, offset):
    shifted = parameters_with_symmetric_com_offset(offset, params)
    print(f"{label}: offset={offset:+.6f}")
    print(f"  lc1/L1={shifted.lc1 / shifted.l1:.6f}, lc2/L2={shifted.lc2 / shifted.l2:.6f}")
    print(f"  lc1={shifted.lc1:.6f} m, lc2={shifted.lc2:.6f} m")

print("Base uniform-link parameters before applying any COM offset:")
print(params)
print(f"gamma = {np.rad2deg(GAMMA):.1f} deg")
print("COM convention: lc1 is measured support -> elbow; lc2 is measured elbow -> free/contact end.")
print("Symmetric offset rule: lc1/L1 = 0.5 + offset, lc2/L2 = 0.5 - offset.")
print_com_offset("center baseline", 0.0)
print_com_offset("reference most-stable COM from the main sweep", REFERENCE_BEST_COM_OFFSET)
for special_offset in REFERENCE_SPECIAL_OFFSETS:
    print_com_offset("special-point comparison", special_offset)


## 2. Find the Baseline Fixed Gait

The local COM scan needs a reliable baseline gait at offset `0`. This cell finds the same stride-distance fixed point used by the main notebook, using the current sub-`dt` impact root finder.

In [ ]:
stride_search = scan_stride_fixed_points(
    model=model,
    slope=slope,
    initial_support_point=INITIAL_SUPPORT,
    dt=shoot_dt,
    t_max=shoot_t_max,
    d_bounds=(0.05, 0.95 * (params.l1 + params.l2)),
    d_scan_points=30,
    branches=("negative",),
    periods=(1,),
    initial_direction=initial_direction,
    impact_direction=impact_direction,
    switch_policy=switch_policy,
    collision_mode=CollisionMode.FULL_GRAB_1D,
)
selected_trial = stride_search.selected_trial
if selected_trial is None:
    raise RuntimeError("No baseline fixed gait found.")

d_fixed = float(selected_trial.d)
selected_branch = selected_trial.branch
selected_period = int(selected_trial.period)

print(f"baseline branch = {selected_branch}, period = {selected_period}")
print(f"d_fixed = {d_fixed:.12f} m")
print(f"rho = {selected_trial.spectral_radius:.6f}")

## 3. A. Fine Local Scan Near `-0.35`

This scan first walks from offset `0` to `-0.34` to get a local warm start, then scans the interval `[-0.34, -0.36]` with a small fixed parameter step. If the eigenvalue varies continuously through `+1` or `-1`, the jump was probably undersampled. If it jumps abruptly even here, the map/branch likely changed discontinuously or Newton jumped branches.

In [ ]:
LOCAL_START = -0.34
LOCAL_END = -0.36
LOCAL_STEP = -0.00025
FOLD_TOL = 7.5e-2

def model_with_com_offset(com_offset):
    return TwoLinkBrachiationModel(parameters_with_symmetric_com_offset(com_offset, params))

def make_stride_map_for_offset(com_offset):
    return make_passive_brachiation_stride_map(
        model=model_with_com_offset(com_offset),
        slope=slope,
        dt=shoot_dt,
        t_max=shoot_t_max,
        collision_mode=CollisionMode.FULL_GRAB_1D,
        initial_direction=initial_direction,
        impact_direction=impact_direction,
        branch=selected_branch,
        support_point=INITIAL_SUPPORT,
        switch_policy=switch_policy,
    )

def make_feasibility(_parameter):
    return make_passive_brachiation_feasibility_check(params, dim=1)

# Approach the local window from the known center branch.
approach_offsets = np.linspace(0.0, LOCAL_START, 41)
approach_result = continue_fixed_point_branch(
    P_factory=make_stride_map_for_offset,
    parameters=approach_offsets,
    x0=np.array([d_fixed], dtype=float),
    dim=1,
    feasibility_factory=make_feasibility,
    tol=1e-6,
    max_iter=10,
    delta=1e-5,
    damping=0.8,
    compute_stability=True,
    stop_on_failure=True,
)
if approach_result.stopped_early:
    print("Approach stopped early:", approach_result.stop_reason)

seed_point = next(point for point in reversed(approach_result.points) if point.converged)
local_offsets = np.arange(LOCAL_START, LOCAL_END + 0.5 * LOCAL_STEP, LOCAL_STEP)
local_result = continue_fixed_point_branch(
    P_factory=make_stride_map_for_offset,
    parameters=local_offsets,
    x0=seed_point.x,
    dim=1,
    feasibility_factory=make_feasibility,
    tol=1e-7,
    max_iter=12,
    delta=1e-5,
    damping=0.8,
    compute_stability=True,
    stop_on_failure=False,
)

local_rows = []
for point in local_result.points:
    eig_real = np.nan if point.eigenvalues is None else float(np.real(point.eigenvalues[0]))
    fold_indicator = np.nan if point.fold_indicator is None else float(point.fold_indicator)
    local_rows.append({
        "offset": float(point.parameter),
        "d": float(point.x[0]),
        "converged": point.converged,
        "residual": point.residual_norm,
        "eig_real": eig_real,
        "rho": np.nan if point.spectral_radius is None else float(point.spectral_radius),
        "fold_indicator": fold_indicator,
        "fold_candidate": bool(np.isfinite(fold_indicator) and fold_indicator <= FOLD_TOL),
        "failure_reason": point.failure_reason,
    })

valid_rows = [row for row in local_rows if row["converged"]]
print(f"local scan points = {len(local_rows)}, converged = {len(valid_rows)}")
if local_result.stopped_early:
    print("local stopped early:", local_result.stop_reason)
for row in local_rows[:5] + local_rows[-5:]:
    print(row)

In [ ]:
offsets = np.array([row["offset"] for row in valid_rows])
eig_real = np.array([row["eig_real"] for row in valid_rows])
rho = np.array([row["rho"] for row in valid_rows])
fold_indicator = np.array([row["fold_indicator"] for row in valid_rows])
d_values = np.array([row["d"] for row in valid_rows])

fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
axes[0].plot(offsets, eig_real, "o-", markersize=3, linewidth=1.4)
axes[0].axhline(1.0, color="black", linestyle="--", linewidth=1, label="+1 fold condition")
axes[0].axhline(-1.0, color="tab:purple", linestyle="--", linewidth=1, label="-1 period-doubling")
axes[0].set_ylabel("real eigenvalue")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="best")

axes[1].plot(offsets, rho, "o-", markersize=3, linewidth=1.4, color="tab:orange")
axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[1].set_ylabel("spectral radius")
axes[1].grid(True, alpha=0.3)

axes[2].plot(offsets, d_values, "o-", markersize=3, linewidth=1.4, color="tab:green")
axes[2].set_xlabel("symmetric COM offset")
axes[2].set_ylabel("fixed stride d [m]")
axes[2].grid(True, alpha=0.3)

candidate_mask = np.isfinite(fold_indicator) & (fold_indicator <= FOLD_TOL)
for ax in axes:
    for x in offsets[candidate_mask]:
        ax.axvline(x, color="black", linestyle=":", linewidth=0.9, alpha=0.7)

plt.suptitle("Fine COM scan near suspected special point")
plt.tight_layout()
plt.show()

def nonconverged_runs(rows):
    runs = []
    start = None
    for idx, row in enumerate(rows):
        if not row["converged"] and start is None:
            start = idx
        if start is not None and (row["converged"] or idx == len(rows) - 1):
            end = idx - 1 if row["converged"] else idx
            runs.append((start, end))
            start = None
    return runs

def skipped_nonconverged_between(rows, left_offset, right_offset):
    left_idx = next(i for i, row in enumerate(rows) if row["offset"] == left_offset)
    right_idx = next(i for i, row in enumerate(rows) if row["offset"] == right_offset)
    lo, hi = sorted((left_idx, right_idx))
    return sum(not row["converged"] for row in rows[lo + 1:hi])

if len(offsets) >= 2:
    eig_jump = np.abs(np.diff(eig_real))
    eig_jump_idx = int(np.nanargmax(eig_jump))
    print("largest eig jump:")
    print(f"  between {offsets[eig_jump_idx]:+.6f} and {offsets[eig_jump_idx + 1]:+.6f}")
    print(f"  eig {eig_real[eig_jump_idx]:+.6f} -> {eig_real[eig_jump_idx + 1]:+.6f}, jump={eig_jump[eig_jump_idx]:.6f}")
    print(f"  rho {rho[eig_jump_idx]:.6f} -> {rho[eig_jump_idx + 1]:.6f}")

    d_jump = np.abs(np.diff(d_values))
    d_jump_idx = int(np.nanargmax(d_jump))
    skipped = skipped_nonconverged_between(
        local_rows,
        offsets[d_jump_idx],
        offsets[d_jump_idx + 1],
    )
    parameter_gap = abs(offsets[d_jump_idx + 1] - offsets[d_jump_idx])
    print("largest d jump:")
    print(f"  between {offsets[d_jump_idx]:+.6f} and {offsets[d_jump_idx + 1]:+.6f}")
    print(f"  d {d_values[d_jump_idx]:.9f} -> {d_values[d_jump_idx + 1]:.9f}, jump={d_jump[d_jump_idx]:.9f} m")
    print(f"  parameter gap={parameter_gap:.6f}, skipped nonconverged points={skipped}")

fail_runs = nonconverged_runs(local_rows)
if fail_runs:
    longest = max(fail_runs, key=lambda item: item[1] - item[0] + 1)
    print("nonconverged gaps:")
    print(f"  total gaps={len(fail_runs)}, longest consecutive nonconverged points={longest[1] - longest[0] + 1}")
    for start, end in fail_runs:
        first = local_rows[start]
        last = local_rows[end]
        print(
            f"  {first['offset']:+.6f} to {last['offset']:+.6f}: "
            f"{end - start + 1} points, first reason={first['failure_reason']}"
        )
else:
    print("nonconverged gaps: none")


## 4. B. Full Free-Run Comparison Across the Suspected Jump

Use two offsets on either side of the suspected transition. The defaults match the proposed diagnostic points: `-0.34` and `-0.358`.

In [ ]:
COMPARE_OFFSETS = [-0.34, -0.358]
FREE_RUN_DURATION = 8.0
OFFSET_SNAP_WARNING_TOL = 2.0 * abs(LOCAL_STEP)

def fixed_point_at_offset(offset):
    nearby = min(valid_rows, key=lambda row: abs(row["offset"] - offset))
    offset_error = abs(nearby["offset"] - offset)
    if offset_error > OFFSET_SNAP_WARNING_TOL:
        print(
            "WARNING: requested offset was snapped to a distant converged point: "
            f"requested={offset:+.6f}, used={nearby['offset']:+.6f}, "
            f"delta={offset_error:.6f}, tolerance={OFFSET_SNAP_WARNING_TOL:.6f}"
        )
    return nearby

def simulate_offset(offset):
    fp = fixed_point_at_offset(offset)
    offset_error = abs(fp["offset"] - offset)
    local_model = model_with_com_offset(fp["offset"])
    q0 = ik_from_stride_distance(
        fp["d"],
        slope=slope,
        parameters=local_model.p,
        direction=initial_direction,
        branch=selected_branch,
    )
    samples = simulate(
        model=local_model,
        slope=slope,
        initial_state=BrachiationState(q=q0, qd=np.zeros(2), support_index=0),
        initial_support_point=INITIAL_SUPPORT,
        duration=FREE_RUN_DURATION,
        dt=shoot_dt,
        switch_policy=switch_policy,
        collision_mode=CollisionMode.FULL_GRAB_1D,
    )
    history = samples_to_arrays(samples, slope=slope)
    rel_idx = release_indices(samples)
    impact_count = int(np.count_nonzero(history["phase"] == "impact"))
    release_count = int(np.count_nonzero(history["phase"] == "release"))
    legality = evaluate_elbow_below_slope_section(samples, slope=slope)
    return {
        "requested_offset": offset,
        "offset": fp["offset"],
        "offset_error": offset_error,
        "fixed_point": fp,
        "model": local_model,
        "q0": q0,
        "samples": samples,
        "history": history,
        "release_indices": rel_idx,
        "impact_count": impact_count,
        "release_count": release_count,
        "legality": legality,
    }

runs = [simulate_offset(offset) for offset in COMPARE_OFFSETS]
for run in runs:
    fp = run["fixed_point"]
    rel_idx = run["release_indices"]
    release_q = run["history"]["q"][rel_idx]
    release_qd = run["history"]["qd"][rel_idx]
    print("=== offset", f"{run['offset']:+.6f}", "===")
    print(f"requested offset = {run['requested_offset']:+.6f}, snap delta = {run['offset_error']:.6f}")
    print(f"d* = {fp['d']:.9f}, eig = {fp['eig_real']:+.6f}, rho = {fp['rho']:.6f}")
    print(f"impacts = {run['impact_count']}, releases = {run['release_count']}")
    print(f"max elbow signed distance = {run['legality'].max_signed_distance:+.3e}")
    print("first release q:", np.round(release_q[:5], 6))
    print("first release qd:", np.round(release_qd[:5], 6))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
colors = ["tab:blue", "tab:red"]

for run, color in zip(runs, colors):
    h = run["history"]
    label = f"offset={run['offset']:+.4f}"
    axes[0, 0].plot(h["q"][:, 0], h["qd"][:, 0], color=color, linewidth=1.2, label=label)
    axes[0, 1].plot(h["q"][:, 1], h["qd"][:, 1], color=color, linewidth=1.2, label=label)
    axes[1, 0].plot(h["frees"][:, 0], h["frees"][:, 1], color=color, linewidth=1.2, label=label)
    axes[1, 1].plot(h["times"], h["free_dist"], color=color, linewidth=1.2, label=label)

axes[0, 0].set_xlabel("q1 [rad]")
axes[0, 0].set_ylabel("q1_dot [rad/s]")
axes[0, 0].set_title("q1 phase portrait")
axes[0, 1].set_xlabel("q2 [rad]")
axes[0, 1].set_ylabel("q2_dot [rad/s]")
axes[0, 1].set_title("q2 phase portrait")

all_frees = np.vstack([run["history"]["frees"] for run in runs])
y_grid = np.linspace(all_frees[:, 0].min() - 0.05, all_frees[:, 0].max() + 0.05, 200)
axes[1, 0].plot(y_grid, -y_grid * np.tan(GAMMA), color="black", linewidth=1.0, linestyle="--", label="slope")
axes[1, 0].set_xlabel("y [m]")
axes[1, 0].set_ylabel("z [m]")
axes[1, 0].set_title("free-end trajectory")
axes[1, 0].set_aspect("equal", adjustable="box")

axes[1, 1].axhline(0.0, color="black", linewidth=1.0, linestyle="--")
axes[1, 1].set_xlabel("time [s]")
axes[1, 1].set_ylabel("free-end signed distance [m]")
axes[1, 1].set_title("contact guard signal")

for ax in axes.flat:
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

plt.tight_layout()
plt.show()

## 5. Global Two-Arm Scan: Resolve the Fold's Stable and Unstable Branches

Warm-started natural continuation follows only one arm of a saddle-node and cannot round the nose, so the companion unstable branch never shows up in the COM sweep. Here we instead run an independent global multi-start residual scan (`find_fixed_points_1d`) at each offset over the full stride range, classify every root by its Poincare multiplier `|lambda|`, and plot the two short-stride arms together with the separate long-stride family.

If the stable arm (`|lambda| -> 1` from below) and an unstable arm (`|lambda| -> 1` from above) approach each other in `d` and vanish together, the suspected `-0.35` jump is a genuine fold, and the unstable arm is exactly the branch the continuation could not trace.

In [ ]:
from passive_brachiation import find_fixed_points_1d, poincare_jacobian_eigenvalues_1d

# Each offset is scanned independently over all of d-space, so BOTH arms of the
# fold are recovered without any warm-started continuation. Reuses
# make_stride_map_for_offset / make_feasibility defined in the fine-scan cell.
ARM_OFFSET_START = -0.30
ARM_OFFSET_END = -0.3525
ARM_OFFSET_STEP = -0.0025
D_SCAN_BOUNDS = (0.06, 0.95 * (params.l1 + params.l2))
D_SCAN_POINTS = 70
ROOT_RESIDUAL_TOL = 1e-6
DEGENERATE_RHO = 50.0   # discard near-zero-stride spurious roots (near-singular IK/Jacobian)
JACOBIAN_DELTA = 1e-5

arm_offsets = np.arange(ARM_OFFSET_START, ARM_OFFSET_END + 0.5 * ARM_OFFSET_STEP, ARM_OFFSET_STEP)
arm_records = []
for offset in arm_offsets:
    P = make_stride_map_for_offset(offset)
    feasibility = make_feasibility(offset)
    roots, _ = find_fixed_points_1d(
        P,
        bounds=D_SCAN_BOUNDS,
        num_scan_points=D_SCAN_POINTS,
        feasibility_check=feasibility,
        residual_tol=ROOT_RESIDUAL_TOL,
    )
    for root in roots:
        try:
            _, _, rho = poincare_jacobian_eigenvalues_1d(
                P, root.x, delta=JACOBIAN_DELTA, feasibility_check=feasibility
            )
        except Exception:
            rho = np.nan
        arm_records.append({
            "offset": float(offset),
            "d": float(root.x),
            "rho": float(rho),
            "stable": bool(np.isfinite(rho) and rho < 1.0),
            "degenerate": bool(np.isfinite(rho) and rho >= DEGENERATE_RHO),
        })

physical_records = [r for r in arm_records if not r["degenerate"]]
print(f"offsets scanned = {len(arm_offsets)}, non-degenerate roots = {len(physical_records)}")
for offset in arm_offsets:
    here = sorted(
        (r for r in physical_records if abs(r["offset"] - offset) < 1e-9),
        key=lambda r: r["d"],
    )
    listing = ", ".join(
        f"d={r['d']:.4f}(rho={r['rho']:.3f},{'S' if r['stable'] else 'U'})" for r in here
    )
    print(f"offset {offset:+.4f}: {len(here)} root(s) -> {listing}")

In [ ]:
# Plot the two short-stride arms (the fold) and the separate long-stride family.
D_SPLIT = 0.45  # stride value separating the short-stride family from the long-stride family

def _sorted_xy(records):
    rec = sorted(records, key=lambda r: r["offset"])
    return (
        np.array([r["offset"] for r in rec]),
        np.array([r["d"] for r in rec]),
        np.array([r["rho"] for r in rec]),
    )

stable_short = [r for r in physical_records if r["d"] < D_SPLIT and r["stable"]]
unstable_short = [r for r in physical_records if r["d"] < D_SPLIT and not r["stable"]]
long_family = [r for r in physical_records if r["d"] >= D_SPLIT]

xs_s, ds_s, rho_s = _sorted_xy(stable_short)
xs_u, ds_u, rho_u = _sorted_xy(unstable_short)
xs_l, ds_l, _ = _sorted_xy(long_family)

fig, ax = plt.subplots(figsize=(9.5, 6))
colorbar_source = None
if len(xs_s):
    ax.plot(xs_s, ds_s, "-", color="0.6", linewidth=1.2, zorder=1)
    colorbar_source = ax.scatter(
        xs_s, ds_s, c=rho_s, cmap="viridis_r", s=70, edgecolors="black",
        linewidths=0.5, zorder=3, label="short stable arm (|lambda|<1)",
    )
if len(xs_u):
    ax.plot(xs_u, ds_u, "--", color="tab:red", linewidth=1.2, zorder=1)
    ax.scatter(
        xs_u, ds_u, marker="x", s=80, color="tab:red", linewidths=2, zorder=3,
        label="short unstable arm (|lambda|>1)",
    )
if len(xs_l):
    ax.scatter(
        xs_l, ds_l, marker="s", s=30, facecolors="none", edgecolors="0.5", zorder=2,
        label="separate long-stride family",
    )

# Fold nose: most negative offset at which both short arms still coexist.
common_offsets = sorted(set(np.round(xs_s, 6)) & set(np.round(xs_u, 6)))
if common_offsets:
    nose_offset = min(common_offsets)
    nose_d = float(ds_s[np.argmin(np.abs(xs_s - nose_offset))])
    ax.axvline(nose_offset, color="black", linestyle=":", linewidth=1.1)
    ax.annotate(
        f"both arms coexist down to\noffset={nose_offset:+.4f}\n(fold just beyond)",
        xy=(nose_offset, nose_d),
        xytext=(12, 20),
        textcoords="offset points",
        arrowprops=dict(arrowstyle="->", color="black"),
        fontsize=9,
    )

ax.set_xlabel("symmetric COM offset")
ax.set_ylabel("fixed stride distance d [m]")
ax.set_title("Fold near -0.35: short stable + unstable arms vs separate long-stride family")
ax.grid(True, alpha=0.3)
ax.legend(loc="best")
if colorbar_source is not None:
    cbar = fig.colorbar(colorbar_source, ax=ax)
    cbar.set_label("Poincare spectral radius |lambda| on the stable arm")
plt.tight_layout()
plt.show()

if common_offsets:
    print(f"Two short-stride arms coexist for offsets >= {min(common_offsets):+.4f} (fold just beyond).")
    print("Stable arm |lambda| -> 1 from below, unstable arm |lambda| -> 1 from above => saddle-node fold.")
else:
    print("No coexisting short-stride arm pair found in the scanned window.")

## 6. Interpretation Notes

- If the fine scan shows `eig_real` crossing `+1` or `-1` smoothly, the original jump was likely undersampling.
- If the fine scan still shows a sharp jump over one tiny step, inspect the full free-runs to see whether impact/release counts, phase portraits, or geometry change abruptly.
- If the two full free-runs have different release/impact structure, the eigenvalue jump is likely tied to a physical orbit reconstruction rather than just numerical noise.
- The global two-arm scan (section 5) settles it: the short-stride branch has a stable arm and an unstable companion arm that converge in `d` while their multipliers approach `+1` from opposite sides, then annihilate near offset `-0.352`. That is a genuine saddle-node fold; the unstable arm (below the stable one in `d`) is the branch the warm-started COM sweep could not trace. The `d ~ 0.59` long-stride solutions are a separate family, not the fold's companion.